In [1]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path
import matplotlib.pyplot as plt

sys.path.insert(0 , './../MAIN/')
from network_functions import (
    build_phenotype_KG,
    attach_phenotype_flags_from_df,
    normalize_edges_similarity_from_node_attr,
    fix_pheno_onehot_in_graph,
    knn_threshold,
    distances_to_similarity
)

datadir = "/work/gr-fe/bryan/data/SHCS/"

netdir = Path(datadir) / "Networks"
netdir.mkdir(parents=True, exist_ok=True)

prefix = 'KGshcs'
out_train = netdir / f"{prefix}.gpickle"

WEIGHT_COL = 'Weight'

# Load data
with open(f"{datadir}/02_processed/phenotype.processed.pkl", "rb") as file:
    data = pickle.load(file)

# Filter min phecode counts and patient counts (same logic as notebook)
min_count = 1
phecode_to_keep = data['Phenotype'].value_counts()[data['Phenotype'].value_counts() >= min_count].index
data = data[data['Phenotype'].isin(phecode_to_keep)]

# Build graph
print('Building KG')
G = build_phenotype_KG(data , id_col = 'ID' , phenotype_col = 'Phenotype', weight_col=WEIGHT_COL)
print(G.number_of_nodes(), G.number_of_edges())

# Normalize edges (set chosen metric as weight)
if WEIGHT_COL is None : 
    G = normalize_edges_similarity_from_node_attr(G, metric='cosine', out_attr='weight', replace_weight=True, use_edge_weight_if_available=True)
else :
    _ = distances_to_similarity(G, distance_attr="weight", method="exponential",
                                  tau=None, out_attr="weight", set_as_weight=True)
    
# Attach flags/attributes
G = attach_phenotype_flags_from_df(G, data)

# Fix node attribute in place
count = fix_pheno_onehot_in_graph(G)
print(f"Converted 'phenotype_onehot' to numpy arrays for {count} nodes.")

# Quick sanity check
try:
    any_type = next(iter(G.nodes(data=True)))[1].get("phenotype_onehot", None)
    print(f"Example 'phenotype_onehot' type: {type(any_type)}")
except StopIteration:
    print("Graph has no nodes to check.")

# Threshold edges 
G_train , knn = knn_threshold(G , k=20 , kind='similarity' , mode='union')
print(G_train.number_of_nodes(), G_train.number_of_edges())

# Save results
print(f"Saving knowledge graph to {out_train}")
with open(out_train, 'wb') as f:
    pickle.dump(G_train, f, pickle.HIGHEST_PROTOCOL)

print("Done.")


Building KG
2078 740276
Converted 'phenotype_onehot' to numpy arrays for 0 nodes.
Example 'phenotype_onehot' type: <class 'NoneType'>
2078 25463
Saving knowledge graph to /work/gr-fe/bryan/data/SHCS/Networks/KGshcs.gpickle
Done.


In [2]:
G_train.nodes(data=True)

NodeDataView({'10082': {'pheno_onehot': array([0, 0, 0, 1, 0], dtype=uint8), 'phenotype_weights': {'Healthy': 3758.0}, 'pheno_Healthy': True}, '10087': {'pheno_onehot': array([0, 0, 1, 0, 0], dtype=uint8), 'phenotype_weights': {'DMT2': -5368.0}, 'pheno_DMT2': True}, '10128': {'pheno_onehot': array([1, 1, 1, 0, 0], dtype=uint8), 'phenotype_weights': {'CAD': 93.0, 'CKD': 4240.0, 'DMT2': -2457.0}, 'pheno_CAD': True, 'pheno_CKD': True, 'pheno_DMT2': True}, '10157': {'pheno_onehot': array([0, 0, 0, 1, 0], dtype=uint8), 'phenotype_weights': {'Healthy': 3706.0}, 'pheno_Healthy': True}, '10188': {'pheno_onehot': array([0, 0, 0, 1, 0], dtype=uint8), 'phenotype_weights': {'Healthy': 5306.0}, 'pheno_Healthy': True}, '10190': {'pheno_onehot': array([1, 0, 0, 0, 1], dtype=uint8), 'phenotype_weights': {'CAD': 190.0, 'OST': 6023.0}, 'pheno_CAD': True, 'pheno_OST': True}, '10251': {'pheno_onehot': array([1, 1, 0, 0, 0], dtype=uint8), 'phenotype_weights': {'CAD': 360.0, 'CKD': 981.0}, 'pheno_CAD': True